# Semantic Link – Semantic Model Diff / Change Tracker

**Author:** jve  
**Date:** April 2026  
**Category:** Community Contests  

---

## Abstract

When developing Power BI semantic models it is very hard to answer the question: **"What changed?"**  
There is no built-in `git diff` equivalent for semantic models — after an update cycle you have no structured  
way to see which tables were added, which measures were modified, or which relationships were removed.

This tool solves that pain point by:
1. **Snapshotting** the full metadata of a semantic model (tables, columns, measures, relationships,  
   partitions, roles, and shared expressions) to a versioned JSON file stored in the Fabric Lakehouse.
2. **Comparing** any two snapshots and producing a structured, human-readable diff — categorising every  
   change as *Added*, *Removed*, or *Modified*.
3. **Displaying** the diff as colour-coded DataFrames so teams can review changes at a glance before  
   promoting models to production.

The tool is fully reusable: just set the `DATASET` and `WORKSPACE` variables and run. No hardcoded IDs  
or credentials required — authentication is handled automatically by Fabric.

## Prerequisites

| Requirement | Details |
|---|---|
| **Fabric workspace** | Premium / Fabric-enabled workspace with a semantic model |
| **Lakehouse attached** | A default Lakehouse must be attached to this notebook (snapshots are stored there) |
| **XMLA read endpoint** | Must be enabled on the capacity (required by Semantic Link's TOM connector) |
| **Libraries** | `semantic-link-sempy` (pre-installed in Fabric), `semantic-link-labs` |

### External Libraries Used
- [`semantic-link-labs`](https://github.com/microsoft/semantic-link-labs) – provides the `connect_semantic_model` context manager for TOM access  
- [`sempy.fabric`](https://learn.microsoft.com/en-us/python/api/semantic-link-sempy/sempy.fabric) – core Semantic Link API  

Install `semantic-link-labs` if it is not already present in your environment:

In [ ]:
%pip install semantic-link-labs --quiet

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# ── Data & display ─────────────────────────────────────────────────────────────
import pandas as pd
from IPython.display import display, HTML

# ── Semantic Link ──────────────────────────────────────────────────────────────
import sempy.fabric as fabric
from sempy_labs.tom import connect_semantic_model

print("Libraries loaded successfully.")

---
## ⚙️ Configuration

Set the parameters below before running the rest of the notebook.  
All other cells are driven by these values — nothing else needs to be changed.

In [ ]:
# ── User configuration ─────────────────────────────────────────────────────────
# Name or GUID of the semantic model to snapshot / compare.
DATASET: str = "My Semantic Model"

# Workspace name or GUID. Leave as None to use the notebook's default workspace.
WORKSPACE: Optional[str] = None

# Optional: a short label to attach to each snapshot (e.g. a sprint tag or PR number).
# Leave as None to use the current UTC timestamp only.
SNAPSHOT_LABEL: Optional[str] = None  # e.g. "before-sprint-42"

# Directory inside the attached Lakehouse /Files/ area where snapshots are persisted.
SNAPSHOT_DIR: str = "/lakehouse/default/Files/model_snapshots"
# ──────────────────────────────────────────────────────────────────────────────

---
## 📸 Snapshot Engine

The snapshot engine captures a complete picture of the semantic model's metadata using the **Tabular Object Model (TOM)**  
exposed by `semantic-link-labs`. The snapshot is a plain JSON document so it can be diffed, stored in version control,  
or compared across environments.

**Captured objects:**
- Tables (with type, hidden flag, description, data category, source expression)
- Columns (with data type, format string, expression for calculated columns, display folder, summarize-by)
- Measures (with DAX expression, format string, display folder, KPI presence)
- Relationships (multiplicity, active flag, cross-filter direction)
- Partitions (mode, source type, query/expression)
- Roles (RLS row filters per table)
- Shared Expressions / M parameters

In [ ]:
def capture_snapshot(
    dataset: str,
    workspace: Optional[str] = None,
    label: Optional[str] = None,
) -> Dict[str, Any]:
    """Capture the full metadata of a semantic model as a dictionary.

    Uses the Tabular Object Model (TOM) via ``connect_semantic_model`` to read
    every relevant property across tables, columns, measures, relationships,
    partitions, roles, and shared expressions.

    Args:
        dataset: Name or GUID of the semantic model.
        workspace: Workspace name or GUID. Defaults to the notebook's workspace.
        label: Optional human-readable tag embedded in the snapshot metadata.

    Returns:
        A dictionary representing the model snapshot, ready for JSON serialisation.
    """
    captured_at = datetime.now(timezone.utc).isoformat()

    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        model = tom.model

        snapshot: Dict[str, Any] = {
            "metadata": {
                "dataset": model.Name,
                "workspace": workspace or "(default)",
                "captured_at": captured_at,
                "label": label or "",
            },
            "tables": [],
            "columns": [],
            "measures": [],
            "relationships": [],
            "partitions": [],
            "roles": [],
            "expressions": [],
        }

        # ── Tables ─────────────────────────────────────────────────────────────
        for t in model.Tables:
            is_calc_group = bool(t.CalculationGroup)
            try:
                is_calc_table = tom.is_calculated_table(table_name=t.Name)
            except Exception:
                is_calc_table = False

            t_type = (
                "Calculation Group" if is_calc_group
                else "Calculated Table" if is_calc_table
                else "Table"
            )

            snapshot["tables"].append({
                "name": t.Name,
                "description": t.Description or "",
                "hidden": bool(t.IsHidden),
                "data_category": t.DataCategory or "",
                "type": t_type,
            })

            # ── Columns ────────────────────────────────────────────────────────
            for c in t.Columns:
                if str(c.Type) == "RowNumber":
                    continue  # internal row-number column – not user-visible
                snapshot["columns"].append({
                    "table": t.Name,
                    "name": c.Name,
                    "description": c.Description or "",
                    "hidden": bool(c.IsHidden),
                    "data_type": str(c.DataType),
                    "format_string": c.FormatString or "",
                    "expression": getattr(c, "Expression", "") or "",
                    "display_folder": c.DisplayFolder or "",
                    "sort_by_column": getattr(c, "SortByColumn", None) and c.SortByColumn.Name or "",
                    "summarize_by": str(c.SummarizeBy),
                    "data_category": c.DataCategory or "",
                    "column_type": str(c.Type),
                })

            # ── Measures ───────────────────────────────────────────────────────
            for m in t.Measures:
                snapshot["measures"].append({
                    "table": t.Name,
                    "name": m.Name,
                    "description": m.Description or "",
                    "hidden": bool(m.IsHidden),
                    "expression": m.Expression or "",
                    "format_string": m.FormatString or "",
                    "display_folder": m.DisplayFolder or "",
                    "has_kpi": m.KPI is not None,
                })

            # ── Partitions ─────────────────────────────────────────────────────
            for p in t.Partitions:
                source = p.Source
                src_type = str(type(source).__name__) if source else ""
                query_or_expr = ""
                if source is not None:
                    for attr in ("Query", "Expression", "EntityName"):
                        val = getattr(source, attr, None)
                        if val:
                            query_or_expr = str(val)
                            break
                snapshot["partitions"].append({
                    "table": t.Name,
                    "name": p.Name,
                    "mode": str(p.Mode),
                    "source_type": src_type,
                    "query_or_expression": query_or_expr,
                })

        # ── Relationships ──────────────────────────────────────────────────────
        for r in model.Relationships:
            snapshot["relationships"].append({
                "name": r.Name,
                "from_table": r.FromTable.Name,
                "from_column": r.FromColumn.Name,
                "to_table": r.ToTable.Name,
                "to_column": r.ToColumn.Name,
                "active": bool(r.IsActive),
                "cross_filter_direction": str(r.CrossFilteringBehavior),
                "from_cardinality": str(r.FromCardinality),
                "to_cardinality": str(r.ToCardinality),
            })

        # ── Roles ──────────────────────────────────────────────────────────────
        for role in model.Roles:
            table_permissions = []
            for tp in role.TablePermissions:
                table_permissions.append({
                    "table": tp.Name,
                    "filter_expression": tp.FilterExpression or "",
                })
            snapshot["roles"].append({
                "name": role.Name,
                "description": role.Description or "",
                "table_permissions": table_permissions,
            })

        # ── Shared Expressions (M parameters / Power Query definitions) ────────
        for expr in model.Expressions:
            snapshot["expressions"].append({
                "name": expr.Name,
                "description": expr.Description or "",
                "expression": expr.Expression or "",
                "kind": str(expr.Kind),
            })

    return snapshot


def save_snapshot(
    snapshot: Dict[str, Any],
    snapshot_dir: str = SNAPSHOT_DIR,
) -> str:
    """Persist a snapshot dictionary as a JSON file in the Lakehouse.

    The file name encodes the dataset name, UTC timestamp, and optional label
    so snapshots are naturally ordered chronologically.

    Args:
        snapshot: Snapshot dictionary produced by :func:`capture_snapshot`.
        snapshot_dir: Absolute path to the Lakehouse Files directory.

    Returns:
        The absolute path of the saved JSON file.
    """
    os.makedirs(snapshot_dir, exist_ok=True)

    dataset_name = snapshot["metadata"]["dataset"]
    ts = snapshot["metadata"]["captured_at"].replace(":", "-").replace("+", "Z")
    label_part = f"_{snapshot['metadata']['label']}" if snapshot["metadata"]["label"] else ""

    safe_name = "".join(c if c.isalnum() or c in "-_" else "_" for c in dataset_name)
    filename = f"{safe_name}{label_part}_{ts}.json"
    filepath = os.path.join(snapshot_dir, filename)

    with open(filepath, "w", encoding="utf-8") as fh:
        json.dump(snapshot, fh, indent=2, ensure_ascii=False)

    print(f"Snapshot saved → {filepath}")
    return filepath


def load_snapshot(filepath: str) -> Dict[str, Any]:
    """Load a previously saved snapshot from a JSON file.

    Args:
        filepath: Absolute path to the snapshot JSON file.

    Returns:
        The snapshot dictionary.
    """
    with open(filepath, "r", encoding="utf-8") as fh:
        return json.load(fh)


def list_snapshots(snapshot_dir: str = SNAPSHOT_DIR) -> pd.DataFrame:
    """List all available snapshots in the snapshot directory.

    Args:
        snapshot_dir: Directory to scan for snapshot JSON files.

    Returns:
        A DataFrame with one row per snapshot, sorted by capture time (newest first).
    """
    rows = []
    if not os.path.isdir(snapshot_dir):
        print(f"Snapshot directory not found: {snapshot_dir}")
        return pd.DataFrame()

    for fname in sorted(os.listdir(snapshot_dir), reverse=True):
        if not fname.endswith(".json"):
            continue
        fpath = os.path.join(snapshot_dir, fname)
        try:
            meta = json.load(open(fpath, encoding="utf-8"))["metadata"]
            rows.append({
                "Dataset": meta.get("dataset"),
                "Label": meta.get("label") or "—",
                "Captured At (UTC)": meta.get("captured_at"),
                "File": fpath,
            })
        except Exception as exc:
            rows.append({"Dataset": "ERROR", "Label": str(exc), "Captured At (UTC)": "", "File": fpath})

    return pd.DataFrame(rows)

---
## 🔍 Diff Engine

The diff engine compares two snapshots and categorises every change across all object types into:

| Symbol | Meaning |
|---|---|
| ✅ **Added** | Object exists in the *new* snapshot but not in the *baseline* |
| ❌ **Removed** | Object exists in the *baseline* but not in the *new* snapshot |
| ✏️ **Modified** | Object exists in both but one or more properties differ |
| ✔️ **Unchanged** | Object is identical in both snapshots |

Each object type uses a **natural key** so renames are correctly detected as a *Remove + Add* pair.

In [ ]:
# ── Key definitions per object type ───────────────────────────────────────────
# Each entry: (section_name, key_fields, compare_fields)
_OBJECT_SPECS: List[Tuple[str, List[str], List[str]]] = [
    (
        "tables",
        ["name"],
        ["description", "hidden", "data_category", "type"],
    ),
    (
        "columns",
        ["table", "name"],
        ["description", "hidden", "data_type", "format_string", "expression",
         "display_folder", "sort_by_column", "summarize_by", "data_category", "column_type"],
    ),
    (
        "measures",
        ["table", "name"],
        ["description", "hidden", "expression", "format_string", "display_folder", "has_kpi"],
    ),
    (
        "relationships",
        ["name"],
        ["from_table", "from_column", "to_table", "to_column",
         "active", "cross_filter_direction", "from_cardinality", "to_cardinality"],
    ),
    (
        "partitions",
        ["table", "name"],
        ["mode", "source_type", "query_or_expression"],
    ),
    (
        "roles",
        ["name"],
        ["description", "table_permissions"],
    ),
    (
        "expressions",
        ["name"],
        ["description", "expression", "kind"],
    ),
]


def _diff_section(
    old_items: List[Dict],
    new_items: List[Dict],
    key_fields: List[str],
    compare_fields: List[str],
) -> Dict[str, List]:
    """Compare two lists of metadata objects and return a structured diff.

    Args:
        old_items: List of objects from the baseline snapshot.
        new_items: List of objects from the new snapshot.
        key_fields: Fields that together form the natural key (identity) of an object.
        compare_fields: Fields to compare for modifications.

    Returns:
        Dictionary with keys ``added``, ``removed``, ``modified``,
        each containing a list of affected objects with change details.
    """
    def _make_key(item: Dict) -> tuple:
        return tuple(str(item.get(k, "")) for k in key_fields)

    old_map = {_make_key(i): i for i in old_items}
    new_map = {_make_key(i): i for i in new_items}

    old_keys = set(old_map)
    new_keys = set(new_map)

    added = [new_map[k] for k in sorted(new_keys - old_keys)]
    removed = [old_map[k] for k in sorted(old_keys - new_keys)]

    modified = []
    for k in sorted(old_keys & new_keys):
        field_changes: Dict[str, Dict] = {}
        for field in compare_fields:
            old_val = old_map[k].get(field)
            new_val = new_map[k].get(field)
            if old_val != new_val:
                field_changes[field] = {"old": old_val, "new": new_val}
        if field_changes:
            identity = {f: old_map[k][f] for f in key_fields}
            modified.append({**identity, "changes": field_changes})

    return {"added": added, "removed": removed, "modified": modified}


def compare_snapshots(
    baseline: Dict[str, Any],
    new: Dict[str, Any],
) -> Dict[str, Dict]:
    """Compare two model snapshots and return a full structured diff.

    Args:
        baseline: The *before* snapshot (produced by :func:`capture_snapshot`).
        new: The *after* snapshot.

    Returns:
        A dictionary keyed by object type (``tables``, ``columns``, etc.),
        each value being a dict with ``added``, ``removed``, and ``modified`` lists.
    """
    diff: Dict[str, Dict] = {}
    for section, key_fields, compare_fields in _OBJECT_SPECS:
        diff[section] = _diff_section(
            old_items=baseline.get(section, []),
            new_items=new.get(section, []),
            key_fields=key_fields,
            compare_fields=compare_fields,
        )
    return diff


def diff_summary(diff: Dict[str, Dict]) -> pd.DataFrame:
    """Produce a one-row-per-object-type summary of a diff.

    Args:
        diff: Output of :func:`compare_snapshots`.

    Returns:
        A summary DataFrame with Added / Removed / Modified counts.
    """
    rows = []
    for section in diff:
        d = diff[section]
        total = len(d["added"]) + len(d["removed"]) + len(d["modified"])
        rows.append({
            "Object Type": section.capitalize(),
            "✅ Added": len(d["added"]),
            "❌ Removed": len(d["removed"]),
            "✏️ Modified": len(d["modified"]),
            "Total Changes": total,
        })
    return pd.DataFrame(rows)

---
## 📊 Reporting & Display

Helper functions that transform the raw diff dictionary into readable, styled DataFrames  
and an optional HTML report that can be saved or shared.

In [ ]:
# ── Colour palette for change types ───────────────────────────────────────────
_COLOURS = {
    "Added": "background-color: #d4edda; color: #155724;",
    "Removed": "background-color: #f8d7da; color: #721c24;",
    "Modified": "background-color: #fff3cd; color: #856404;",
}


def _style_row(row):
    colour = _COLOURS.get(row.get("Change", ""), "")
    return [colour] * len(row)


def _flatten_diff_section(
    section_diff: Dict[str, List],
    key_fields: List[str],
    compare_fields: List[str],
) -> pd.DataFrame:
    """Flatten a single section's diff into a tabular DataFrame.

    Args:
        section_diff: The ``{added, removed, modified}`` dict for one object type.
        key_fields: Identity fields for this object type.
        compare_fields: Tracked property fields.

    Returns:
        A flat DataFrame with a ``Change`` column and one row per changed object
        (or one row per changed *property* for modified objects).
    """
    rows = []

    for item in section_diff["added"]:
        row = {"Change": "Added"}
        row.update({f: item.get(f, "") for f in key_fields})
        for f in compare_fields:
            row[f] = item.get(f, "")
        row["Property Changed"] = ""
        row["Old Value"] = ""
        row["New Value"] = ""
        rows.append(row)

    for item in section_diff["removed"]:
        row = {"Change": "Removed"}
        row.update({f: item.get(f, "") for f in key_fields})
        for f in compare_fields:
            row[f] = item.get(f, "")
        row["Property Changed"] = ""
        row["Old Value"] = ""
        row["New Value"] = ""
        rows.append(row)

    for item in section_diff["modified"]:
        for prop, vals in item["changes"].items():
            row = {"Change": "Modified"}
            row.update({f: item.get(f, "") for f in key_fields})
            row["Property Changed"] = prop
            row["Old Value"] = str(vals["old"]) if vals["old"] is not None else ""
            row["New Value"] = str(vals["new"]) if vals["new"] is not None else ""
            rows.append(row)

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    # Drop compare_fields columns — they're only useful for Added/Removed rows as context
    cols = ["Change"] + key_fields + ["Property Changed", "Old Value", "New Value"]
    return df[[c for c in cols if c in df.columns]]


def display_diff(
    diff: Dict[str, Dict],
    baseline_meta: Optional[Dict] = None,
    new_meta: Optional[Dict] = None,
) -> None:
    """Interactively display a full diff with colour-coded tables per object type.

    Unchanged object types are skipped. Call :func:`diff_summary` first for a quick
    overview if you only want high-level counts.

    Args:
        diff: Output of :func:`compare_snapshots`.
        baseline_meta: Optional metadata dict from the baseline snapshot (for headers).
        new_meta: Optional metadata dict from the new snapshot (for headers).
    """
    if baseline_meta and new_meta:
        display(HTML(
            f"<h3>🔄 Diff: <code>{baseline_meta.get('label') or baseline_meta.get('captured_at','baseline')}</code> "
            f"→ <code>{new_meta.get('label') or new_meta.get('captured_at','new')}</code></h3>"
            f"<p><b>Dataset:</b> {baseline_meta.get('dataset')}  "
            f"| <b>Baseline captured:</b> {baseline_meta.get('captured_at')}  "
            f"| <b>New captured:</b> {new_meta.get('captured_at')}</p>"
        ))

    has_changes = False
    for section, key_fields, compare_fields in _OBJECT_SPECS:
        sec_diff = diff.get(section, {})
        total = len(sec_diff.get("added", [])) + len(sec_diff.get("removed", [])) + len(sec_diff.get("modified", []))
        if total == 0:
            continue

        has_changes = True
        df = _flatten_diff_section(sec_diff, key_fields, compare_fields)
        if df.empty:
            continue

        added_count = len(sec_diff["added"])
        removed_count = len(sec_diff["removed"])
        modified_count = len(sec_diff["modified"])

        display(HTML(
            f"<h4 style='margin-top:20px;margin-bottom:4px'>📦 {section.capitalize()} "
            f"<span style='font-size:0.85em;font-weight:normal;color:#555'> "
            f"(+{added_count} added | -{removed_count} removed | ~{modified_count} modified)</span></h4>"
        ))

        styled = (
            df.style
            .apply(lambda row: [_COLOURS.get(row["Change"], "")] * len(row), axis=1)
            .hide(axis="index")
        )
        display(styled)

    if not has_changes:
        display(HTML("<p style='color:green;font-weight:bold'>✔️ No changes detected — the two snapshots are identical.</p>"))


def export_diff_html(
    diff: Dict[str, Dict],
    output_path: str,
    baseline_meta: Optional[Dict] = None,
    new_meta: Optional[Dict] = None,
) -> str:
    """Export the diff as a self-contained HTML report file.

    Args:
        diff: Output of :func:`compare_snapshots`.
        output_path: Absolute path for the HTML report (e.g. inside Lakehouse Files).
        baseline_meta: Optional metadata from the baseline snapshot.
        new_meta: Optional metadata from the new snapshot.

    Returns:
        The path of the saved HTML file.
    """
    parts = ["<html><head><meta charset='utf-8'>",
             "<style>body{font-family:sans-serif;padding:20px} table{border-collapse:collapse;width:100%} ",
             "th,td{border:1px solid #ccc;padding:6px 10px;text-align:left;font-size:0.9em} ",
             "th{background:#f0f0f0}</style></head><body>"]

    if baseline_meta and new_meta:
        parts.append(
            f"<h2>Semantic Model Diff: {baseline_meta.get('dataset')}</h2>"
            f"<p><b>Baseline:</b> {baseline_meta.get('label') or baseline_meta.get('captured_at')} &nbsp;→&nbsp; "
            f"<b>New:</b> {new_meta.get('label') or new_meta.get('captured_at')}</p>"
        )

    summary_df = diff_summary(diff)
    parts.append("<h3>Summary</h3>" + summary_df.to_html(index=False))

    for section, key_fields, compare_fields in _OBJECT_SPECS:
        sec_diff = diff.get(section, {})
        total = len(sec_diff.get("added", [])) + len(sec_diff.get("removed", [])) + len(sec_diff.get("modified", []))
        if total == 0:
            continue
        df = _flatten_diff_section(sec_diff, key_fields, compare_fields)
        if df.empty:
            continue

        def _colour_row(row):
            colour_map = {"Added": "#d4edda", "Removed": "#f8d7da", "Modified": "#fff3cd"}
            bg = colour_map.get(row["Change"], "white")
            return [f"background-color: {bg}" for _ in row]

        html_table = df.style.apply(_colour_row, axis=1).hide(axis="index").to_html()
        parts.append(f"<h3>{section.capitalize()}</h3>{html_table}")

    parts.append("</body></html>")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as fh:
        fh.write("".join(parts))

    print(f"HTML report saved → {output_path}")
    return output_path

---
## 🚀 Workflow

The typical usage pattern is:

```
Step 1  →  Take a baseline snapshot  (before making model changes)
Step 2  →  Edit the model in Power BI Desktop / Fabric Model View
Step 3  →  Take a new snapshot                                     
Step 4  →  Compare the two snapshots and review the diff           
```

Run the cells below in sequence the first time. On subsequent runs only the
cells relevant to your current step are needed.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 – Take a snapshot of the current model state.
#
# Run this cell BEFORE making changes to capture the baseline.
# Run it again AFTER changes to capture the updated state.
# ─────────────────────────────────────────────────────────────────────────────

print(f"Capturing snapshot for dataset: '{DATASET}' ...")
snapshot = capture_snapshot(dataset=DATASET, workspace=WORKSPACE, label=SNAPSHOT_LABEL)
snapshot_path = save_snapshot(snapshot, snapshot_dir=SNAPSHOT_DIR)

print(f"\nSnapshot summary:")
for section in ["tables", "columns", "measures", "relationships", "partitions", "roles", "expressions"]:
    print(f"  {section.capitalize():15}: {len(snapshot[section])} objects")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 – List available snapshots.
#
# Shows all JSON snapshots in SNAPSHOT_DIR, newest first.
# Use the "File" paths to load specific snapshots in Step 3.
# ─────────────────────────────────────────────────────────────────────────────

snapshots_df = list_snapshots(SNAPSHOT_DIR)
if snapshots_df.empty:
    print("No snapshots found. Run Step 1 to create your first snapshot.")
else:
    display(snapshots_df.style.hide(axis="index"))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 – Compare two snapshots.
#
# Set BASELINE_FILE and NEW_FILE to the paths shown by Step 2.
# You can also compare live model state vs. a saved snapshot:
#   new_snapshot = capture_snapshot(dataset=DATASET, workspace=WORKSPACE)
# ─────────────────────────────────────────────────────────────────────────────

# ── Set these to the two snapshot files you want to compare ───────────────────
BASELINE_FILE: str = ""  # e.g. "/lakehouse/default/Files/model_snapshots/MyModel_2026-04-06.json"
NEW_FILE: str = ""       # leave empty to use the snapshot captured in Step 1 above
# ──────────────────────────────────────────────────────────────────────────────

if not BASELINE_FILE:
    # Auto-select the two most recent snapshots if both paths are unset
    available = list_snapshots(SNAPSHOT_DIR)
    if len(available) < 2:
        print("⚠️  Need at least 2 snapshots to compare. Run Step 1 a second time after editing the model.")
    else:
        BASELINE_FILE = available.iloc[1]["File"]
        NEW_FILE = available.iloc[0]["File"]
        print(f"Auto-selected:\n  Baseline: {BASELINE_FILE}\n  New:      {NEW_FILE}")

if BASELINE_FILE and NEW_FILE:
    baseline_snapshot = load_snapshot(BASELINE_FILE)
    new_snapshot = load_snapshot(NEW_FILE) if NEW_FILE != snapshot_path else snapshot
    diff = compare_snapshots(baseline=baseline_snapshot, new=new_snapshot)
    print("\nDiff computed. Run the next cell to view results.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 – View the diff.
# ─────────────────────────────────────────────────────────────────────────────

# ── High-level summary (counts per object type) ───────────────────────────────
display(HTML("<h3>📋 Change Summary</h3>"))
summary = diff_summary(diff)
display(
    summary.style
    .hide(axis="index")
    .bar(subset=["Total Changes"], color="#90caf9")
)

# ── Detailed colour-coded diff per object type ────────────────────────────────
display_diff(
    diff,
    baseline_meta=baseline_snapshot["metadata"],
    new_meta=new_snapshot["metadata"],
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# OPTIONAL – Export the diff as a shareable HTML report.
# ─────────────────────────────────────────────────────────────────────────────

report_path = os.path.join(SNAPSHOT_DIR, "diff_report.html")
export_diff_html(
    diff=diff,
    output_path=report_path,
    baseline_meta=baseline_snapshot["metadata"],
    new_meta=new_snapshot["metadata"],
)

---
## 🔜 Next Steps & Ideas for Extension

| Idea | Details |
|---|---|
| **CI/CD gate** | Run this notebook in a Fabric pipeline triggered on model publish; fail the pipeline if unexpected objects were removed |
| **Slack / Teams alert** | Post the change summary to a channel via webhook after each deployment |
| **Git integration** | Save snapshot JSON directly to a Git repository alongside model `.pbip` files for true diff history |
| **Cross-environment diff** | Compare a Dev and Prod workspace snapshot to validate a promotion |
| **Scheduled baseline** | Schedule a daily snapshot notebook run to build a full change audit trail |
| **DAX expression diff** | Use a line-diff algorithm (e.g. `difflib`) to highlight DAX changes inline rather than as full expression replacement |

---

### Libraries & References

- [`semantic-link-labs`](https://github.com/microsoft/semantic-link-labs) – TOM wrapper and helper functions  
- [`sempy.fabric`](https://learn.microsoft.com/en-us/python/api/semantic-link-sempy/sempy.fabric) – Semantic Link core API  
- [Microsoft Fabric Notebook Docs](https://learn.microsoft.com/en-us/fabric/data-engineering/how-to-use-notebook)  
- [Tabular Object Model Reference](https://learn.microsoft.com/en-us/analysis-services/tom/introduction-to-the-tabular-object-model-tom-in-analysis-services-amo)